In [ ]:
import requests, time, html, csv
from xml.etree import ElementTree as ET

In [ ]:
import requests, time, html
from xml.etree import ElementTree as ET

pmids = [
    36597288, 35397762, 36575200, 36531928, 35743993, 36454572, 35266128,
    34838369, 34893431, 35999019, 36352367, 35059902, 36504604, 35888568,
    35032409, 35072716, 35953265, 35208539, 36427058, 36429392, 35908724,
    36429368, 35223629, 35831582, 34378840, 34432058, 35971558, 36192799,
    36412591,
]

def fetch_abstracts(pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(map(str, pmids)),
        "retmode": "xml",
        "rettype": "abstract",
    }
    xml = requests.get(url, params=params).text
    root = ET.fromstring(xml)
    results = []
    for art in root.findall(".//PubmedArticle"):
        pmid = art.findtext(".//PMID")
        title = art.findtext(".//ArticleTitle") or ""
        parts = [html.unescape(node.text or "") for node in art.findall(".//AbstractText")]
        abstract = "\n".join(p for p in parts if p)
        results.append((pmid, title.strip(), abstract.strip()))
    return results

records = fetch_abstracts(pmids)
for r in records:
    print(f"PMID {r[0]}:\n{r[1]}\n{r[2]}\n{'-'*60}\n")
    time.sleep(0.3)


In [ ]:
import requests, html
from xml.etree import ElementTree as ET
import pandas as pd

pmids = [
    36597288, 35397762, 36575200, 36531928, 35743993, 36454572, 35266128,
    34838369, 34893431, 35999019, 36352367, 35059902, 36504604, 35888568,
    35032409, 35072716, 35953265, 35208539, 36427058, 36429392, 35908724,
    36429368, 35223629, 35831582, 34378840, 34432058, 35971558, 36192799,
    36412591,
]

def fetch_abstracts(pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(map(str, pmids)),
        "retmode": "xml",
        "rettype": "abstract",
    }
    xml = requests.get(url, params=params).text
    root = ET.fromstring(xml)
    data = []
    for art in root.findall(".//PubmedArticle"):
        pmid = art.findtext(".//PMID")
        title = art.findtext(".//ArticleTitle") or ""
        parts = [html.unescape(node.text or "") for node in art.findall(".//AbstractText")]
        abstract = " ".join(p.strip() for p in parts if p)
        data.append({"PMID": pmid, "Title": title.strip(), "Abstract": abstract})
    return pd.DataFrame(data)

# Fetch data
df = fetch_abstracts(pmids)

# Save to Excel
output_file = "abstracts.xlsx"
df.to_excel(output_file, index=False)

print(f"✅ Saved {len(df)} abstracts to {output_file}")
